# Interview Preparation - KnowBe4

## Create fake data set

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random


# Faker helps generate realistic fake data
fake = Faker()
# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)


# Number of users
n_users = 500
# Create user IDs
user_ids = np.arange(1, n_users + 1)
# Possible countries
countries = [
    "Netherlands",
    "Germany",
    "South Africa",
    "USA",
    "UK",
    "France"
]
# Generate user dataset
users = pd.DataFrame({
    "user_id": user_ids,
    # Random country assignment
    "country": np.random.choice(countries, n_users),
    # Simulate account age in days
    "account_age_days": np.random.randint(1, 3000, n_users)
})
# Display first few rows
users.head()

# Number of events
n_events = 20000
# Event types
event_types = [
    "login",
    "logout",
    "file_download",
    "password_reset",
    "email_click",
    "vpn_access"
]
# Create empty list to store events
event_rows = []
# Generate event rows
for _ in range(n_events):
    # Random user
    user_id = np.random.choice(user_ids)
    # Random timestamp within last 30 days
    timestamp = (
        datetime.now() - timedelta(
            days=np.random.randint(0, 30),
            hours=np.random.randint(0, 24),
            minutes=np.random.randint(0, 60)
        )
    )
    # Random event type
    event_type = np.random.choice(event_types)
    # Generate fake IP address
    ip_address = fake.ipv4_public()
    # Simulate highly imbalanced target
    # About 2% threats
    is_threat = np.random.choice(
        [0, 1],
        p=[0.98, 0.02]
    )
    # Add suspicious behavior patterns
    # Threats more likely to happen at night
    if is_threat == 1:
        timestamp = timestamp.replace(
            hour=np.random.choice([1, 2, 3, 4])
        )
    # Store row
    event_rows.append([
        user_id,
        timestamp,
        event_type,
        ip_address,
        is_threat
    ])
# Create events DataFrame
events = pd.DataFrame(
    event_rows,
    columns=[
        "user_id",
        "timestamp",
        "event_type",
        "ip_address",
        "is_threat"
    ]
)
# Display first rows
events.head()

# Introduce some missing user IDs
missing_indices = np.random.choice(events.index, 100)
events.loc[missing_indices, "user_id"] = np.nan
# Introduce missing IP addresses
missing_ip_indices = np.random.choice(events.index, 50)
events.loc[missing_ip_indices, "ip_address"] = np.nan

# Save datasets
users.to_parquet("users.parquet", index=False)
events.to_parquet("events.parquet", index=False)
print("Parquet files successfully created!")

Parquet files successfully created!


## Load data into DF's

In [2]:
events = pd.read_parquet('events.parquet')
users = pd.read_parquet('users.parquet')

In [3]:
print(events.info())
print("")
print(users.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     19902 non-null  float64       
 1   timestamp   20000 non-null  datetime64[ns]
 2   event_type  20000 non-null  object        
 3   ip_address  19950 non-null  object        
 4   is_threat   20000 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 781.4+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id           500 non-null    int64 
 1   country           500 non-null    object
 2   account_age_days  500 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 11.8+ KB
None


## Join DF's

In [4]:
df = pd.merge(
    events,
    users,
    on='user_id',
    how='left'
)
df

,user_id,timestamp,event_type,ip_address,is_threat,country,account_age_days
0,369.0,2026-04-11 17:41:31.653700,file_download,213.135.30.195,0,Netherlands,2682.0
1,194.0,2026-04-27 07:11:31.655456,logout,151.189.104.187,0,UK,1251.0
2,300.0,2026-05-07 11:53:31.656102,vpn_access,132.131.239.173,0,South Africa,268.0
3,212.0,2026-05-10 21:30:31.656149,password_reset,185.197.136.233,0,Netherlands,1032.0
4,90.0,2026-04-28 11:42:31.656180,vpn_access,165.9.150.227,0,UK,76.0
...,...,...,...,...,...,...,...
19995,480.0,2026-04-26 02:58:32.044627,password_reset,5.15.146.111,0,USA,2398.0
19996,277.0,2026-04-18 14:41:32.044643,file_download,217.114.213.144,0,Netherlands,714.0
19997,22.0,2026-04-17 03:37:32.044665,password_reset,201.200.174.33,0,Netherlands,2208.0
19998,100.0,2026-04-15 19:50:32.044686,vpn_access,158.196.89.224,0,UK,396.0


## Inspect df

In [5]:
df.head(20)

,user_id,timestamp,event_type,ip_address,is_threat,country,account_age_days
0,369.0,2026-04-11 17:41:31.653700,file_download,213.135.30.195,0,Netherlands,2682.0
1,194.0,2026-04-27 07:11:31.655456,logout,151.189.104.187,0,UK,1251.0
2,300.0,2026-05-07 11:53:31.656102,vpn_access,132.131.239.173,0,South Africa,268.0
3,212.0,2026-05-10 21:30:31.656149,password_reset,185.197.136.233,0,Netherlands,1032.0
4,90.0,2026-04-28 11:42:31.656180,vpn_access,165.9.150.227,0,UK,76.0
5,356.0,2026-04-29 07:16:31.656207,login,211.157.200.71,0,France,1995.0
6,10.0,2026-05-06 19:09:31.656244,password_reset,212.131.104.48,0,UK,2490.0
7,168.0,2026-05-09 22:18:31.656274,email_click,11.155.38.47,0,USA,1959.0
8,265.0,2026-05-01 02:55:31.656896,vpn_access,218.45.47.119,0,USA,385.0
9,94.0,2026-04-25 12:19:31.656925,password_reset,84.14.126.143,0,Netherlands,718.0


In [6]:
df['is_threat'].value_counts(normalize=True)

is_threat
0    0.9811
1    0.0189
Name: proportion, dtype: float64

## Clean data
- Important is to signal nulls before filling them
- Nulls in user_id and ip_address might be most interesting

In [7]:
# Signal missing data
df['missing_userid'] = df['user_id'].isna().astype(int)
df['missing_ip'] = df['ip_address'].isna().astype(int)

In [8]:
# Fill nulls in user_id column
df['user_id'] = df['user_id'].fillna(0).astype(int)

# Make sure timestamp is in correct datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Make sure ip address is string and fillna
df['ip_address'] = df['ip_address'].fillna('unknown')
df['ip_address'] = df['ip_address'].astype('string')

# Fill missing country values
df['country'] = df['country'].fillna('unknown')

# Fill missing account age values
df['account_age_days'] = df['account_age_days'].fillna(0)

## Feature Engineering
- Temporarily use timestamp as index for rolling window
- Add features for strange times, like on weekend or at night
- Add time features to check behaviour changes
- Look for behaviour signals in time related events, also in ip_addresses

In [9]:
# Sort values firts
df = df.sort_values(['user_id', 'timestamp'])

# In order to create rolling time window, timestamp needs to be index
df = df.set_index('timestamp')

In [14]:
# Time related features - behaviour
df['is_weekend'] = df.index.day_of_week.isin([5, 6]).astype(int)
df['is_night'] = df.index.hour.isin([0, 1, 2, 3, 4, 5]).astype(int)

# Time related rolling window behaviour
df['user_events_24hrs'] = (
    df.groupby('user_id')['user_id']
      .rolling('24h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

df['user_events_12hrs'] = (
    df.groupby('user_id')['user_id']
      .rolling('12h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

df['user_events_1hr'] = (
    df.groupby('user_id')['user_id']
      .rolling('1h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

In [ ]:
# IP related features
# Count each users total IP addresses over a 24hr period
# Factorize IP column first
df['_ip_code'] = pd.factorize(df['ip_address'])[0]

# Check unique IP addresses per user over a 24hr period
df['unique_ips_24hr'] = (
    df.groupby('user_id')['_ip_code']
      .rolling('24h')
      .apply(lambda x: pd.Series(x).nunique(), raw=False)
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)



,user_id,event_type,ip_address,is_threat,country,account_age_days,missing_userid,missing_ip,is_weekend,is_night,user_events_24hrs,user_events_12hrs,user_events_1hr,_ip_code,unique_ips_24hr
timestamp,,,,,,,,,,,,,,,
2026-04-12 00:15:31.730931,0,password_reset,34.133.236.121,0,unknown,0.0,1,0,1,1,0,0,0,0,0
2026-04-12 03:43:31.757182,0,file_download,175.153.158.119,0,unknown,0.0,1,0,1,1,1,1,1,1,1
2026-04-12 05:21:32.015375,0,email_click,192.219.89.11,0,unknown,0.0,1,0,1,1,2,2,1,2,2
2026-04-12 12:24:31.691379,0,file_download,171.176.172.186,0,unknown,0.0,1,0,1,0,3,3,1,3,3
2026-04-13 04:24:31.870733,0,email_click,202.92.112.117,0,unknown,0.0,1,0,0,1,4,3,1,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-07 19:46:31.738409,500,login,105.55.181.249,0,USA,118.0,0,0,0,0,3,2,1,19946,3
2026-05-08 03:46:32.009474,500,password_reset,80.117.54.14,0,USA,118.0,0,0,0,1,3,2,1,19947,3
2026-05-09 09:16:31.882131,500,password_reset,135.47.6.105,0,USA,118.0,0,0,1,0,4,3,1,19948,4
